# Resolução Case PS Inteli Academy

## 1. Introdução



Neste notebook, serão desenvolvidos modelos de Machine Learning para prever a rotatividade de clientes (churn) da empresa fictícia TelecomPlus. O objetivo é identificar, com base em dados históricos, quais clientes têm maior probabilidade de cancelar seus serviços nos próximos meses. Para isso, apliquei técnicas de pré-processamento, engenharia de features, codificação de variáveis, balanceamento de classes e treinamento de diferentes algoritmos de classificação. Ao final, os modelos são avaliados e comparados.

## 2. Importação das Principais Bibliotecas

In [ ]:
# Importação das principais bibliotecas usadas para a resolucação do case

import pandas as pd
pd.options.mode.chained_assignment = None

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## 3. Carregamento e Visão Geral dos Dados

In [ ]:
# Carregando o banco de dados

df = pd.read_csv("dados_clientes.csv")

In [ ]:
# Idêntificação do tipo de dados de cada colunas
# Contagem de valores não nulos
df.info()

In [ ]:
# Análise descritiva das colunas numéricas 

df.describe()

### 3.1 Identificação de Valores Nuloes, Duplicados e Iguais a 0

In [ ]:
duplicated = df[df.duplicated()]

print("Valores duplicados: ")
print(len(duplicated))

In [ ]:
# Identificação de valores iguais a 0

zero_count = (df == 0).sum()
print(zero_count)

In [ ]:
df_no_id = df.drop('id_cliente', axis=1)

zero_count = (df_no_id == 0).sum()
print(zero_count)

duplicated = df_no_id[df_no_id.duplicated()]

print("Valores duplicados: ")
print(len(duplicated))

print("\n\nMissing Values: \n")
print(df_no_id.isnull().sum())

In [ ]:
df_no_id_churn = df.drop(columns=['id_cliente', 'churn'])
duplicated = df_no_id_churn[df_no_id_churn.duplicated()]

print("Valores duplicados:", len(duplicated))


print("\n\nMissing Values:\n", df_no_id_churn.isnull().sum())


De acordo com os outputs acima, não há informações faltantes mascaradas pelo id_cliente e churn


### 3.2 Identificação de outliers

In [ ]:
# Plotando os gráficos que ajudarão a identificar os outliers em cada coluna. Obs: Os outliers são identificados pelos pontos vermelhos

fig, axs = plt.subplots(1, 6, layout='constrained')

sns.boxplot(data=df, y='total_gasto', ax=axs[0], flierprops={'marker':'o', 'markersize':8, 'markerfacecolor':'red'})

sns.boxplot(data=df, y='valor_mensal', ax=axs[1], flierprops={'marker':'o', 'markersize':8, 'markerfacecolor':'red'})

sns.boxplot(data=df, y='tempo_como_cliente', ax=axs[2], flierprops={'marker':'o', 'markersize':8, 'markerfacecolor':'red'})

sns.boxplot(data=df, y='suporte_contatado', ax=axs[3], flierprops={'marker':'o', 'markersize':8, 'markerfacecolor':'red'})

sns.boxplot(data=df, y='tempo_medio_atendimento', ax=axs[4], flierprops={'marker':'o', 'markersize':8, 'markerfacecolor':'red'})

sns.boxplot(data=df, y='chamados_abertos', ax=axs[5], flierprops={'marker':'o', 'markersize':8, 'markerfacecolor':'red'})

plt.show()

## 4. Tratamento de Dados


In [ ]:
df_tratada = df.copy()

### 4.1 Tratamento de Outliears

In [ ]:
# Removento outliers

df_tratada = df_tratada[df_tratada['total_gasto'] <= 100001].reset_index(drop=True)
df_tratada = df_tratada[df_tratada['suporte_contatado'] < 8].reset_index(drop=True)
df_tratada = df_tratada[df_tratada['chamados_abertos'] < 6].reset_index(drop=True)

### 4.2 Tratamento de Missing Values

**OBS: Decidi seguir pelo caminho de apenas remover as linhas com valores zerados, pois elas continham a lista de produtos vazia. Enfrentei bastante dificuldade para preencher essas listas de forma artificial e correta utilizando alguma estratégia de imputação. Por isso, considerei melhor simplesmente excluir essas linhas.**

In [ ]:
df_tratada['produtos_assinados'].value_counts()

In [ ]:
# Remove as 7650 linhas que apresentavam as features abaixo zeradas

df_tratada = df_tratada[~((df_tratada['servicos_assinados'] == 0) & 
                            (df_tratada['valor_mensal'] == 0) & 
                            (df_tratada['total_gasto'] == 0))]

### 4.3 Ajuste de formatação de dados

In [ ]:
# Ajusta a formatação dos dados na coluna 'produtos_assinados' para garantir que os nomes dos produtos estejam corretamente separados
df_tratada['produtos_assinados'] = df_tratada['produtos_assinados'].apply(lambda x: x.replace("' ", "',"))


In [ ]:
# Arredonda os valores das colunas 'valor_mensal' e 'total_gasto' para 2 casas decimais
# Isso ajuda a padronizar os dados financeiros e evita excesso de precisão desnecessária
df_tratada[['valor_mensal', 'total_gasto']] = df_tratada[['valor_mensal', 'total_gasto']].round(2)


## 5. Eng. de Feature

**Dividi a feature `produtos_assinados` em colunas individuais para cada produto, com o objetivo de identificar padrões específicos associados a cada um deles.**

In [ ]:
# Lista com os nomes dos produtos que queremos extrair da coluna 'produtos_assinados'
produtos = ['Produto A', 'Produto B', 'Produto C', 'Produto D', 'Produto E', 'Produto F']

# Para cada produto da lista acima, cria uma nova coluna no DataFrame com valor 1 se o produto estiver presente e 0 caso contrário
for prod in produtos:
    df_tratada[prod] = df_tratada['produtos_assinados'].apply(lambda x: 1 if prod in x else 0)

# Remove a coluna original 'produtos_assinados' já que agora temos os dados de forma estruturada em colunas separadas
df_tratada = df_tratada.drop('produtos_assinados', axis=1)

**Criei a feature de `faixa_idade` para facilitar a identificação de padrões de churn entre diferentes grupos de idade, reduzir a variabilidade dos dados e melhorar a interpretabilidade e desempenho dos modelos preditivos.**

In [ ]:
# Define as condições para classificar a idade dos clientes em faixas etárias
faixas_idade = [
    (df_tratada['idade'] <= 18),
    (df_tratada['idade'] >= 19) & (df_tratada['idade'] <= 25),
    (df_tratada['idade'] >= 26) & (df_tratada['idade'] <= 30),
    (df_tratada['idade'] >= 31) & (df_tratada['idade'] <= 35),
    (df_tratada['idade'] >= 36) & (df_tratada['idade'] <= 40),
    (df_tratada['idade'] >= 41) & (df_tratada['idade'] <= 45),
    (df_tratada['idade'] >= 46) & (df_tratada['idade'] <= 50),
    (df_tratada['idade'] >= 51) & (df_tratada['idade'] <= 55),
    (df_tratada['idade'] >= 56) & (df_tratada['idade'] <= 60),
    (df_tratada['idade'] >= 61) & (df_tratada['idade'] <= 74),
    (df_tratada['idade'] >= 75)    
]

# Define os rótulos correspondentes para cada faixa etária
choices = ['0-18', '19-25','26-30','31-35','36-40','41-45', '46-50', '51-55', '56-60', '61-74', '75+']

# Cria uma nova coluna 'faixa_idade' com as categorias definidas acima com base nas condições
df_tratada['faixa_idade'] = np.select(faixas_idade, choices, default='Não definido')

# Remove a coluna original de idade, já que ela foi transformada em categorias
df_tratada.drop(columns=['idade'], inplace=True)


**Cria uma nova coluna chamada `atrasos_valor` que representa o impacto financeiro dos atrasos de pagamento.**

In [ ]:
df_tratada['atrasos_valor'] = df_tratada['atrasos_pagamento'] * df_tratada['valor_mensal']

**Alterei a feature `total_gasto` em faixas codificadas para representar os níveis de gasto de forma ordenada, facilitando a interpretação e o uso pelos modelos.**

In [ ]:
# Cria os limites dos bins de 1 até 100000 com intervalos de 50 unidades.
# O valor np.inf é adicionado para capturar qualquer valor acima de 100000.
bins = list(range(1, 100001, 50))
bins.append(np.inf)

# Cria os rótulos para as faixas no mesmo intervalo (50 unidades).
# Exemplo: "1-50", "51-100", ..., até "99951-100000"
labels = [f"{i}-{i+49}" for i in range(1, 100000, 50)]

# Usa pd.cut para categorizar os valores de 'total_gasto' nas faixas criadas.
# right=False garante que os intervalos sejam fechados à esquerda e abertos à direita.
# include_lowest=True assegura que o valor mínimo seja incluído corretamente.
df_tratada['total_gasto_faixa'] = pd.cut(
    df_tratada['total_gasto'], 
    bins=bins, 
    labels=labels, 
    right=False,           
    include_lowest=True
)

# Codifica as categorias em valores numéricos, adicionando +1 para começar do 1 (e não do 0).
df_tratada['total_gasto_encoded'] = df_tratada['total_gasto_faixa'].cat.codes + 1

# Substitui a coluna original 'total_gasto' pela versão codificada.
df_tratada['total_gasto'] = df_tratada['total_gasto_encoded']

# Remove as colunas intermediárias que não são mais necessárias.
df_tratada.drop(columns=['total_gasto_faixa', 'total_gasto_encoded'], inplace=True)


In [ ]:
import re

def to_snake_case(col_name):
    """
    Converte nomes de colunas para o formato snake_case.
    """
    # Substitui espaços por underscore e também elimina underscores duplos, se existirem
    col_name = col_name.replace(' ', '_').replace('__', '_')
    
    # Converte todos os caracteres da string para letras minúsculas
    col_name = col_name.lower()
    
    # Remove qualquer caractere que não seja letra, número ou underscore
    col_name = re.sub(r'[^a-z0-9_]', '', col_name)
    
    return col_name  # Retorna o nome convertido

# Aplica a função de conversão a todas as colunas do DataFrame df_tratada
df_tratada = df_tratada.rename(columns=lambda c: to_snake_case(c))


## 6. Preprocessamento e codificação de variáveis

In [ ]:
df_encoded = df_tratada.copy()

In [ ]:
# Remover as colunas originais que não serão necessárias
df_encoded.drop(columns=['tipo_contrato', 'forma_pagamento', 'estado_civil', 'genero', 'id_cliente'], inplace=True)

# Defina a ordem das faixas, conforme o seu array 'choices'
ordem_idade = ['0-18', '19-25', '26-30', '31-35', '36-40', '41-45', '46-50', '51-55', '56-60', '61-74', '75+']

# Converte a coluna 'faixa_idade' para categoria ordenada
df_encoded['faixa_idade'] = pd.Categorical(df_encoded['faixa_idade'], categories=ordem_idade, ordered=True)

# Cria uma nova coluna com os códigos (ordinal encoding)
df_encoded['faixa_idade'] = df_encoded['faixa_idade'].cat.codes

# Defina a ordem das faixas, conforme o seu array 'choices'
ordem_renda = ['0-1000', '1000-5000', '5000-10000', '10000-50000', '50000-100000']

# Converte a coluna 'faixa_idade' para categoria ordenada
df_encoded['renda_faixa'] = pd.Categorical(df_encoded['renda_faixa'], categories=ordem_renda, ordered=True)

# Cria uma nova coluna com os códigos (ordinal encoding)
df_encoded['renda_faixa'] = df_encoded['renda_faixa'].cat.codes

## 7. Análise Exploratória

In [ ]:
# Cria um dicionário para armazenar a taxa de churn de cada produto
churn_rate = {}

# Lista com os nomes dos produtos
produtos = ['produto_a', 'produto_b', 'produto_c', 'produto_d', 'produto_e', 'produto_f']

# Itera sobre a lista de produtos
for prod in produtos:
    # Para cada produto, calcula a média da coluna 'churn' apenas para os clientes que assinaram esse produto
    churn_rate[prod] = df_tratada.loc[df_tratada[prod] == 1, 'churn'].mean()

# Exibe as taxas de churn calculadas para cada produto
print("Taxa de churn por produto:")
for prod in churn_rate:
    print(f"{prod}: {churn_rate[prod]:.2f}")

# Inicializa um dicionário para armazenar os preços médios estimados de cada produto
precos = {}

print("\n")

# Itera sobre cada produto para calcular o valor médio
for prod in produtos:
    # Define uma condição onde o cliente assinou somente o produto atual
    cond = (df_tratada[prod] == 1)
    for outro in produtos:
        if outro != prod:
            # Garante que os outros produtos não estão ativos
            cond = cond & (df_tratada[outro] == 0)
    
    # Calcula a média da coluna 'valor_mensal' para quem assinou apenas esse produto
    preco_medio = df_tratada.loc[cond, 'valor_mensal'].mean()
    precos[prod] = preco_medio
    print(f"Preço aproximado do {prod} = {preco_medio:.2f}")

media_custos = {
    'Produto A': 50,
    'Produto B': 80,
    'Produto C': 120,
    'Produto D': 200,
    'Produto E': 250,
    'Produto F': 300
}

In [ ]:
df_heatmap = df_encoded.copy()

In [ ]:
# Seleciona só as colunas numéricas (tirando a target)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_heatmap_scaled = scaler.fit_transform(df_heatmap)

# Converte o array escalado de volta para um DataFrame para usar o método .corr()
df_heatmap_scaled = pd.DataFrame(df_heatmap_scaled, columns=df_encoded.columns)

# 2. Criar o heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df_heatmap_scaled.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Mapa de Calor das Correlações Entre Features")
plt.show()

In [ ]:
col_drop = [
    "reclamacoes",
    'produto_a',
    'produto_b',
    'produto_c',
    'produto_d',
    'produto_e',
    'produto_f'
]

df_encoded.drop(columns=col_drop, inplace=True)


In [ ]:
from sklearn.model_selection import train_test_split

# Separe as variáveis independentes e dependentes:
X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

# Divida os dados em treino e teste de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

from sklearn.ensemble import RandomForestClassifier

# Suponha que model seja um RandomForest já ajustado (fit)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Obter a importância das features
importances = model.feature_importances_
features = X_train.columns

# Criar um DataFrame para facilitar a visualização
df_importances = pd.DataFrame({'Feature': features, 'Importance': importances})
df_importances = df_importances.sort_values('Importance', ascending=True)

plt.figure(figsize=(7, 4))
plt.barh(df_importances['Feature'], df_importances['Importance'], color='skyblue')
plt.xlabel("Importância")
plt.title("Importância das Features para Churn")
plt.show()

In [ ]:
# Headmap sem as features menos relevantes

plt.figure(figsize=(10, 6))
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Mapa de Calor - Correlação entre Features")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

# Aplica o Permutation Importance no modelo já treinado `model`
# A técnica embaralha os valores de cada feature e observa o impacto na performance
result = permutation_importance(model, X, y, n_repeats=10, random_state=42)

# Calcula a média da importância para cada feature e organiza em ordem decrescente
importances = pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False)

# Plota as importâncias em um gráfico de barras horizontal
importances.plot(kind='barh', figsize=(7,6), title='Permutation Importance')

# Inverte o eixo Y para que a feature mais importante apareça no topo
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Remove faetures que apresentaram importância praticamente insignificante pelas análises acima
df_encoded.drop(columns=['tempo_medio_atendimento', 'faixa_idade'], inplace=True)


## 8. Treinando Modelos

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

In [ ]:
# Separe as variáveis independentes e dependentes:
X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

# Divida os dados em treino e teste de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

### 8.1 Separação entre modelos normalizados e de árvore

In [ ]:
# Versão original para modelos baseados em árvore
X_tree = X.copy()

# Versão normalizada para modelos sensíveis a escala
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Converte de volta para DataFrame com os mesmos nomes de coluna
X_normalized = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

### 8.2 Funções de treino (com validação cruzada)

In [ ]:
def treinar_modelo_normalizado(modelo, X_normalized, y, test_size=0.3, random_state=42, cv=5):
    # Split tradicional
    X_train, X_test, y_train, y_test = train_test_split(
        X_normalized, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    # Treina o modelo com os dados de treino
    modelo.fit(X_train, y_train)

    # Faz predições no conjunto de teste
    y_pred = modelo.predict(X_test)

    # Exibe métricas de desempenho do modelo
    print("Modelo com Normalização")
    print(classification_report(y_test, y_pred))
    print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))

    # Validação cruzada
    scores = cross_val_score(modelo, X_normalized, y, cv=cv, scoring='f1')
    print(f"\nValidação Cruzada (F1-score): Média = {scores.mean():.4f} | Desvio = {scores.std():.4f}")

    return modelo  # Retorna o modelo treinado

In [ ]:
def treinar_modelo_arvore(modelo, X_tree, y, test_size=0.3, random_state=42, cv=5):
    # Separa os dados em treino e teste de forma estratificada (mantém proporção de classes)
    X_train, X_test, y_train, y_test = train_test_split(
        X_tree, y, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=y
    )

    # Treina o modelo com os dados de treino
    modelo.fit(X_train, y_train)

    # Faz predições no conjunto de teste
    y_pred = modelo.predict(X_test)

    # Exibe métricas de desempenho do modelo
    print("Modelo Baseado em Árvore")
    print(classification_report(y_test, y_pred))          
    print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))

    # Validação cruzada
    scores = cross_val_score(modelo, X_tree, y, cv=cv, scoring='f1')
    print(f"\nValidação Cruzada (F1-score): Média = {scores.mean():.4f} | Desvio = {scores.std():.4f}")

    return modelo  # Retorna o modelo treinado

### 8.3 Treinamento de todos os modelos

In [ ]:
# Importa os modelos que se beneficiam de dados normalizados
from sklearn.linear_model import LogisticRegression  
from sklearn.neighbors import KNeighborsClassifier  

# Lista de modelos que requerem normalização dos dados
modelos_normalizados = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)), 
    ("KNN", KNeighborsClassifier()),
]

In [ ]:
# Importa os modelos de classificação baseados em árvores
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier    
from sklearn.tree import DecisionTreeClassifier 

# Lista de modelos que NÃO necessitam de normalização
modelos_arvore = [
    ("Random Forest", RandomForestClassifier()),

    ("XGBoost", XGBClassifier(use_label_encoder=False, eval_metric='logloss', verbosity=0)),

    ("Decision Tree", DecisionTreeClassifier())
]

In [ ]:
# Para modelos de árvore
for nome, modelo in modelos_arvore:
    print(f"\nAvaliando: {nome}")
    treinar_modelo_arvore(modelo, X_tree, y)

In [ ]:
# Para modelos com normalização
for nome, modelo in modelos_normalizados:
    print(f"\nAvaliando: {nome}")
    treinar_modelo_normalizado(modelo, X_normalized, y)

## 9. Balanceamento de Classes

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.utils import compute_class_weight

In [ ]:
# Calcular pesos das classes
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
class_weights_dict = dict(zip(np.unique(y), class_weights))

# Aplicar SMOTE no conjunto de treino
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Normalizar os dados para os modelos que precisam
scaler = StandardScaler()
X_res_normalized = pd.DataFrame(scaler.fit_transform(X_train_res), columns=X.columns)
X_test_normalized = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

### 9.1 Treinamento de Modelos com Classes balanceadas

In [ ]:
# Funções de treinamento
def treinar_com_balanceamento(modelo, X_train_res, y_train_res, X_test, y_test, nome):
    modelo.fit(X_train_res, y_train_res)
    y_pred = modelo.predict(X_test)

    print(f"\nModelo: {nome}")
    print(classification_report(y_test, y_pred))
    print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))

    scores = cross_val_score(modelo, X_train_res, y_train_res, cv=5, scoring='f1')
    print(f"Validação Cruzada (F1-score): Média = {scores.mean():.4f} | Desvio = {scores.std():.4f}")

# Modelos que precisam de normalização
modelos_norm = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("KNN", KNeighborsClassifier())
]

# Modelos baseados em árvore
modelos_tree = [
    ("Random Forest", RandomForestClassifier(class_weight=class_weights_dict)),
    ("XGBoost", XGBClassifier(use_label_encoder=False, eval_metric='logloss', verbosity=0)),
    ("Decision Tree", DecisionTreeClassifier(class_weight=class_weights_dict))
]

In [ ]:
# Executar treinamento com balanceamento
for nome, modelo in modelos_norm:
    treinar_com_balanceamento(modelo, X_res_normalized, y_train_res, X_test_normalized, y_test, nome)

In [ ]:
for nome, modelo in modelos_tree:
    treinar_com_balanceamento(modelo, X_train_res, y_train_res, X_test, y_test, nome)

Após testar modelos com e sem balanceamento via SMOTE, percebi que o oversampling não trouxe ganhos relevantes. Em alguns casos, até piorou a performance por possível overfitting. Por isso, decidi seguir com modelos que já lidam bem com desbalanceamento, usando `class_weight='balanced'` e `scale_pos_weight` no XGBoost. Essa estratégia deve mantém os dados originais e melhora a detecção de clientes com maior risco de churn.

In [ ]:
from collections import Counter

# Conta quantos exemplos há de cada classe
counts = Counter(y)
ratio = counts[0] / counts[1]


In [ ]:
modelos_sem_oversampling = [
    ("Logistic Regression (balanced)", LogisticRegression(class_weight='balanced', max_iter=1000)),
    ("Random Forest (balanced)", RandomForestClassifier(class_weight='balanced', random_state=42)),
    ("Decision Tree (balanced)", DecisionTreeClassifier(class_weight='balanced')),
    ("XGBoost (scale_pos_weight)", XGBClassifier(scale_pos_weight=ratio, use_label_encoder=False, eval_metric='logloss', verbosity=0))
]


In [ ]:
for nome, modelo in modelos_sem_oversampling:
    treinar_modelo_arvore(modelo, X_tree, y)


## 10. Estratégias Avançadas de Modelagem

### 10.1 Stacking de Modelos


Combinando pontos fortes de diferentes algoritmos para melhorar a generalização.

In [ ]:
from sklearn.ensemble import StackingClassifier

# Define os estimadores base
estimators = [
    ('rf', RandomForestClassifier(class_weight='balanced', random_state=42)),
    ('xgb', XGBClassifier(scale_pos_weight=ratio, use_label_encoder=False, eval_metric='logloss', verbosity=0)),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))
]

# Define o modelo empilhado (stacking)
stacked_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    passthrough=False,
    n_jobs=-1
)

# Treina o modelo empilhado
stacked_model.fit(X_train, y_train)

# Faz as previsões no conjunto de teste
y_pred = stacked_model.predict(X_test)

# Avaliação padrão
print("Modelo Empilhado (StackingClassifier)")
print(classification_report(y_test, y_pred))
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))

# Validação cruzada com todo o dataset
scores = cross_val_score(stacked_model, X_tree, y, cv=5, scoring='f1')
print(f"Validação Cruzada (F1-score): Média = {scores.mean():.4f} | Desvio = {scores.std():.4f}")


### 10.2 LightGBM com Otimização de Hiperparâmetros

In [ ]:
import optuna
from lightgbm import LGBMClassifier


optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "scale_pos_weight": ratio,
        "verbosity": -1
    }

    model = LGBMClassifier(**params)
    score = cross_val_score(model, X, y, cv=5, scoring="f1").mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=False)

print("Melhores parâmetros:", study.best_params)
print("Melhor pontuação (F1):", study.best_value)



### 10.3 Otimização de Threshold para Maximizar Recall


In [ ]:
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import precision_recall_curve
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils import resample

# 1. Feature selection
top_k = 9
selector = SelectKBest(score_func=f_classif, k=top_k)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

# 2. Modelo leve calibrado
modelo_primario = LogisticRegression(max_iter=2000, class_weight='balanced')
modelo_calibrado = CalibratedClassifierCV(estimator=modelo_primario, method='sigmoid', cv=5)
modelo_calibrado.fit(X_train_sel, y_train)

# 3. Probabilidades do modelo leve
probs_primario = modelo_calibrado.predict_proba(X_test_sel)[:, 1]

# 4. Identifica casos ambíguos
lim_inf, lim_sup = 0.4, 0.7
idx_ambiguos = (probs_primario > lim_inf) & (probs_primario < lim_sup)
X_detalhado = X_test[idx_ambiguos]
y_detalhado = y_test[idx_ambiguos]

# 5. Treinamento do modelo detalhado (se houver ambíguos)
if len(X_detalhado) > 0:
    X_detalhado, y_detalhado = resample(X_detalhado, y_detalhado, n_samples=min(5000, len(X_detalhado)), random_state=42)

    modelo_detalhado = XGBClassifier(
        eval_metric='logloss',
        use_label_encoder=False,
        n_estimators=100
    )
    modelo_detalhado.fit(X_detalhado, y_detalhado)

    # 6. Threshold ótimo com base em precisão mínima
    y_probs_detalhado = modelo_detalhado.predict_proba(X_detalhado)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_detalhado, y_probs_detalhado)
    precisions, recalls = precisions[:-1], recalls[:-1]
    min_precision = 0.75
    indices_validos = np.where(precisions >= min_precision)

    if len(indices_validos[0]) > 0:
        valid_thresholds = thresholds[indices_validos]
        valid_recalls = recalls[indices_validos]
        threshold_otimo = valid_thresholds[np.argmax(valid_recalls)]
    else:
        threshold_otimo = 0.5

    print(f"Threshold ótimo (precisão ≥ {min_precision*100:.0f}%): {threshold_otimo:.3f}")

    # 7. Substitui previsões nos ambíguos
    y_pred_final = modelo_calibrado.predict(X_test_sel)
    probs_detalhado_final = modelo_detalhado.predict_proba(X_test[idx_ambiguos])[:, 1]
    y_pred_ambiguos = (probs_detalhado_final >= threshold_otimo).astype(int)
    y_pred_final[idx_ambiguos] = y_pred_ambiguos

    # 8. Avaliação final
    print("\nRelatório Final - Modelo Híbrido (Teste):")
    print(classification_report(y_test, y_pred_final))
    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred_final))
else:
    print("Nenhum caso ambíguo identificado — modelo detalhado não foi treinado.")
    y_pred_final = modelo_calibrado.predict(X_test_sel)

## 11. Avaliação Final e Conclusão

Durante o desenvolvimento deste projeto, foram exploradas diversas abordagens para prever o churn de clientes.

Inicialmente, foram avaliados modelos tradicionais como Random Forest, XGBoost, Decision Tree, Logistic Regression e KNN, aplicados tanto em dados desbalanceados quanto balanceados via SMOTE. Embora o oversampling tenha trazido alguma melhoria na sensibilidade, também introduziu variação nas métricas e, em alguns casos, overfitting.

A estratégia foi então redirecionada para o uso de **modelos que lidam nativamente com desbalanceamento**, utilizando `class_weight='balanced'` e `scale_pos_weight`. Isso trouxe maior estabilidade nos resultados, com destaque para:

- **Random Forest (balanced)**, que apresentou bom desempenho geral e robustez.
- **XGBoost com `scale_pos_weight`**, que melhorou a detecção da classe minoritária.
- **Logistic Regression (balanced)**, com performance estável e interpretabilidade.
- **LightGBM com tuning via Optuna**, que obteve o melhor F1-score médio dentre os modelos testados (~0.564).

Como avanço técnico, foi implementado um **modelo híbrido em cascata**, combinando:

- Um modelo leve (Logistic Regression calibrado com validação cruzada e seleção de features),
- Um modelo mais robusto (XGBoost), aplicado apenas aos casos ambíguos, definidos com base na probabilidade prevista.

Além disso, foi feita a **calibração de probabilidades** e **otimização de threshold** com base na curva precision-recall, buscando manter uma precisão mínima de 75% nos casos ambíguos.

O modelo híbrido, portanto, foi usado como solução final.

## 12. Aplicação no conjunto desafio.csv

In [ ]:
df_desafio = pd.read_csv("desafio.csv")
df_res = df_desafio.copy()

In [ ]:
# Aplicando o mesmo pré-processamento dos dados de treino nos dados do desafio


df_desafio[['valor_mensal', 'total_gasto']] = df_desafio[['valor_mensal', 'total_gasto']].round(2)

faixas_idade = [
    (df_desafio['idade'] <= 18),
    (df_desafio['idade'] >= 19) & (df_desafio['idade'] <= 25),
    (df_desafio['idade'] >= 26) & (df_desafio['idade'] <= 30),
    (df_desafio['idade'] >= 31) & (df_desafio['idade'] <= 35),
    (df_desafio['idade'] >= 36) & (df_desafio['idade'] <= 40),
    (df_desafio['idade'] >= 41) & (df_desafio['idade'] <= 45),
    (df_desafio['idade'] >= 46) & (df_desafio['idade'] <= 50),
    (df_desafio['idade'] >= 51) & (df_desafio['idade'] <= 55),
    (df_desafio['idade'] >= 56) & (df_desafio['idade'] <= 60),
    (df_desafio['idade'] >= 61) & (df_desafio['idade'] <= 74),
    (df_desafio['idade'] >= 75)
]
choices = ['0-18', '19-25','26-30','31-35','36-40','41-45', '46-50', '51-55', '56-60', '61-74', '75+']
df_desafio['faixa_idade'] = np.select(faixas_idade, choices, default='Não definido')

ordem_idade = ['0-18', '19-25', '26-30', '31-35', '36-40', '41-45', '46-50', '51-55', '56-60', '61-74', '75+']
df_desafio['faixa_idade'] = pd.Categorical(df_desafio['faixa_idade'], categories=ordem_idade, ordered=True)
df_desafio['faixa_idade'] = df_desafio['faixa_idade'].cat.codes

df_desafio.drop(columns=['idade'], inplace=True)

df_desafio['atrasos_valor'] = df_desafio['atrasos_pagamento'] * df_desafio['valor_mensal']

bins = list(range(1, 100001, 50))
bins.append(np.inf)

labels = [f"{i}-{i+49}" for i in range(1, 100000, 50)]

df_desafio['total_gasto_faixa'] = pd.cut(
    df_desafio['total_gasto'], 
    bins=bins, 
    labels=labels, 
    right=False,           
    include_lowest=True
)
df_desafio['total_gasto_encoded'] = df_desafio['total_gasto_faixa'].cat.codes + 1
df_desafio['total_gasto'] = df_desafio['total_gasto_encoded']
df_desafio.drop(columns=['total_gasto_faixa', 'total_gasto_encoded'], inplace=True)

# Converte nomes de colunas para snake_case
def to_snake_case(col_name):
    col_name = col_name.replace(' ', '_').replace('__', '_')
    col_name = col_name.lower()
    col_name = re.sub(r'[^a-z0-9_]', '', col_name)
    return col_name

df_desafio = df_desafio.rename(columns=lambda c: to_snake_case(c))

ordem_renda = ['0-1000', '1000-5000', '5000-10000', '10000-50000', '50000-100000']
df_desafio['renda_faixa'] = pd.Categorical(df_desafio['renda_faixa'], categories=ordem_renda, ordered=True)
df_desafio['renda_faixa'] = df_desafio['renda_faixa'].cat.codes

df_desafio.drop(columns=['tipo_contrato', 'forma_pagamento', 'estado_civil', 'genero', 'id_cliente', 'tempo_medio_atendimento', 'produtos_assinados', 'reclamacoes', 'faixa_idade'], inplace=True)


In [ ]:
# 1. Reutiliza o seletor já treinado no conjunto de treino
X_desafio_sel = selector.transform(df_desafio)

# 2. Probabilidades com o modelo leve calibrado
probs_primario_desafio = modelo_calibrado.predict_proba(X_desafio_sel)[:, 1]

# 3. Define o intervalo de ambiguidade (mesmo usado no treino)
lim_inf, lim_sup = 0.4, 0.7
idx_ambiguos_desafio = (probs_primario_desafio > lim_inf) & (probs_primario_desafio < lim_sup)
X_detalhado_desafio = df_desafio[idx_ambiguos_desafio]

# 4. Aplica o modelo detalhado nos dados ambíguos
if len(X_detalhado_desafio) > 0:
    probs_detalhado_desafio = modelo_detalhado.predict_proba(X_detalhado_desafio)[:, 1]
    y_pred_ambiguos_desafio = (probs_detalhado_desafio >= threshold_otimo).astype(int)
else:
    y_pred_ambiguos_desafio = np.array([])

# 5. Predição base com o modelo leve
y_pred_desafio = modelo_calibrado.predict(X_desafio_sel)

# 6. Substitui as predições ambíguas pelo modelo detalhado
y_pred_desafio[idx_ambiguos_desafio] = y_pred_ambiguos_desafio

# 7. Gera o DataFrame final de resultados
resultado_final = pd.DataFrame({
    'Id': df_res['id_cliente'], 
    'Target': y_pred_desafio
})

# 8. Exporta o resultado
resultado_final.to_csv('resultado_roberto_filho.csv', index=False)